# Demo — AgentCore Memory Continuity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-memory-continuity/demo-memory-continuity.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 4: Context and Visibility (Memory and Observability)

Demonstrates how memory provides session continuity without the backend
needing to pass massive message histories on every request. Differentiates
between short-term dialogue history and long-term semantic facts.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json
import os

try:
    import boto3
    HAS_BOTO3 = True
except ImportError:
    HAS_BOTO3 = False

# --- Configuration ---
MEMORY_ID = os.environ.get("MEMORY_ID", "mem-12345")
SESSION_ID = "session-flight-77x"
USER_ID = "user-alice-123"

if HAS_BOTO3:
    memory = boto3.client("bedrock-agentcore-memory")

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def compare_stateless_vs_stateful():
    """Explain the shift from stateless chat API to stateful AgentCore API."""
    print_section("Stateless API vs. Stateful AgentCore")

    stateless = {
        "api": "Converse / Chat API",
        "pattern": "Backend maintains DB, fetches last 10 messages, sends to API",
        "payload_size": "Large (all history sent every time)",
        "bottleneck": "Network latency and backend DB limits",
    }

    stateful = {
        "api": "AgentCore InvokeAgentRuntime",
        "pattern": "Backend sends ONLY the new user utterance + session ID",
        "payload_size": "Tiny (just the new message)",
        "benefit": "AgentCore fetches history natively from Memory within AWS",
    }

    print("  Stateless (Old Way):")
    print(f"  {json.dumps(stateless, indent=2)}")
    print("\n  Stateful (AgentCore Way):")
    print(f"  {json.dumps(stateful, indent=2)}")

def show_short_term_memory():
    """Short-term memory = the dialogue window for THIS session."""
    print_section("Short-Term Memory (Session Context)")

    print(f"  [Memory API] Fetching dialogue for {SESSION_ID}")

    history = [
        {"role": "user", "text": "I need a flight to London next week."},
        {"role": "agent", "text": "I can help. What dates?"},
        {"role": "user", "text": "Leaving Monday, returning Friday."},
    ]

    for msg in history:
        print(f"  {msg['role'].upper():>5}: {msg['text']}")

    print("\n  - Scoped strictly to the sessionId")
    print("  - Automatically truncated or summarized when approaching context limits")

def show_long_term_memory():
    """Long-term memory = facts and preferences across ALL sessions."""
    print_section("Long-Term Memory (User Context)")

    print(f"  [Memory API] Fetching semantic facts for {USER_ID}")

    facts = [
        {"fact": "Prefers aisle seats", "confidence": 0.95},
        {"fact": "Frequent flyer number: BA-98765", "confidence": 1.0},
        {"fact": "Travels with pet (dog)", "confidence": 0.82},
    ]

    for f in facts:
        print(f"  FACT: {f['fact']:<35} (Conf: {f['confidence']})")

    print("\n  - Scoped strictly to the runtimeUserId")
    print("  - Persists across multiple sessions/days/months")
    print("  - Extracted automatically in the background using vector embeddings")

def show_redaction_responsibility():
    """Memory stores data. Redacting PII is the developer's job."""
    print_section("WARNING: PII and Data Residency")

    print("  AgentCore Memory is a data store. It remembers what it is told.")
    print("  If the user says: 'My SSN is 123-45-6789'")
    print("  Memory will store: 'User SSN is 123-45-6789'")
    print()
    print("  Best Practices:")
    print("  1. Use Amazon Comprehend or a regex proxy to redact PII BEFORE it hits AgentCore.")
    print("  2. Provide a 'forget me' API that calls AgentCore DeleteMemory to comply with GDPR.")
    print("  3. Configure KMS customer-managed keys (CMK) for Memory encryption.")

def main():
    print("Memory Continuity — Instructor Demo\n")
    compare_stateless_vs_stateful()
    show_short_term_memory()
    show_long_term_memory()
    show_redaction_responsibility()

    print_section("Key Takeaways")
    print("  1. AgentCore shifts state management from your backend to AWS infrastructure.")
    print("  2. Short-term memory tracks the current conversation (Session ID).")
    print("  3. Long-term memory extracts and retrieves facts across time (User ID).")
    print("  4. PII redaction and compliance (GDPR deletion) remain your responsibility.")

if __name__ == "__main__":
    main()
